<a href="https://colab.research.google.com/github/rathodgayathri9-glitch/neural-networks-machine-learning-exercises/blob/main/Exercise-06-BAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#!/usr/bin/env python3
"""
BAM Handwritten Digit <-> Code Association System
-------------------------------------------------
Dataset : Optical Recognition of Handwritten Digits (UCI optdigits, 8x8, values 0-16)

Run     : python bam_digits.py [optdigits.tra | optdigits.tes | digits.csv]

Data lookup order
  1. file given on the command line (64 pixel columns + label, comma separated)
  2. optdigits.tra / optdigits.tes in the current folder
  3. scikit-learn's bundled copy of the UCI optdigits test set (1797 samples, works offline)
  4. download from UCI

Requires: numpy   (scikit-learn optional, used only as an offline data source)
"""
import os
import sys
import urllib.request

import numpy as np

SEED = 7
UCI_BASE = "https://archive.ics.uci.edu/ml/machine-learning-databases/optdigits/"
CODE_LEN = 16
THRESHOLD = 8          # pixel (0..16) >= 8  ->  +1 ink, else -1 background


# ----------------------------------------------------------------------------
# data
# ----------------------------------------------------------------------------
def _from_csv(path):
    a = np.loadtxt(path, delimiter=",")
    return a[:, :64], a[:, 64].astype(int)


def load_data(path=None):
    for p in ([path] if path else []) + ["optdigits.tra", "optdigits.tes"]:
        if p and os.path.exists(p):
            print(f"[data] Loading {p}")
            X, y = _from_csv(p)
            return X, y, "file: " + p
    try:
        from sklearn.datasets import load_digits
        d = load_digits()
        print("[data] Using scikit-learn bundled UCI optdigits (1797 samples).")
        return d.data.astype(float), d.target.astype(int), "UCI optdigits via scikit-learn"
    except Exception:
        pass
    try:
        print("[data] Downloading optdigits.tra from UCI ...")
        urllib.request.urlretrieve(UCI_BASE + "optdigits.tra", "optdigits.tra")
        X, y = _from_csv("optdigits.tra")
        return X, y, "UCI download"
    except Exception as e:
        sys.exit(f"[data] Could not load a dataset ({e}). Put optdigits.tra next to this script.")


def to_bipolar(X):
    return np.where(X >= THRESHOLD, 1, -1).astype(int)


def hadamard(n):
    H = np.array([[1]])
    while H.shape[0] < n:
        H = np.block([[H, H], [H, -H]])
    return H


def make_codes():
    """10 unique, mutually orthogonal 16-bit bipolar codes (rows 1..10 of a Hadamard matrix)."""
    return hadamard(CODE_LEN)[1:11].astype(int)


def sgn(v, prev=None):
    out = np.where(v > 0, 1, -1)
    if prev is not None:                       # zero net input -> keep previous state
        out = np.where(v == 0, prev, out)
    return out


def show(p, title="", mask=None):
    """Print an 8x8 bipolar pattern. '#' = +1, '.' = -1, ' ' = unknown (0)."""
    if title: print(title)
    g = p.reshape(8, 8)
    for r in g:
        print("   " + " ".join("#" if v > 0 else ("." if v < 0 else " ") for v in r))


def side_by_side(a, b, ta, tb):
    print(f"   {ta:<17}{tb}")
    for r1, r2 in zip(a.reshape(8, 8), b.reshape(8, 8)):
        f = lambda r: " ".join("#" if v > 0 else ("." if v < 0 else " ") for v in r)
        print(f"   {f(r1)}      {f(r2)}")


def code_str(c): return "".join("+" if v > 0 else "-" for v in c)


# ----------------------------------------------------------------------------
# BAM
# ----------------------------------------------------------------------------
class BAM:
    """
    Correlation (Hebbian) BAM with bias thresholds.
      Learning : W = sum_p (x_p - mu)^T y_p   (mu = mean stored pattern, i.e. threshold/bias)
      Forward  : y = sgn((x - mu) W)
      Backward : x = sgn(mu + y W^T / (L * k))
    Centering removes the large common "background" component shared by all digits,
    which otherwise makes a plain x^T y BAM favour a single class.
    """
    def __init__(self, codes):
        self.codes = codes
        self.W = np.zeros((64, codes.shape[1]), dtype=float)
        self.mu = np.zeros(64)
        self.n_pairs = 0
        self.scale = 1.0

    def store(self, Xb, labels):
        self.mu = Xb.mean(0)
        self.W = np.zeros((64, self.codes.shape[1]))
        for x, l in zip(Xb, labels):
            self.W += np.outer(x - self.mu, self.codes[l])
        self.n_pairs = len(labels)
        self.scale = max(1.0, (self.n_pairs / 10) * self.codes.shape[1])

    def _center(self, x): return np.where(x == 0, 0.0, x - self.mu)   # 0 = unknown pixel

    def forward(self, x, prev=None):   return sgn(self._center(x) @ self.W, prev)
    def backward(self, y, prev=None):  return sgn(self.mu + (y @ self.W.T) / self.scale, prev)

    def decode(self, y):
        """Nearest stored code (min Hamming distance). Returns (digit, distance)."""
        d = (self.codes != y).sum(1)
        return int(d.argmin()), int(d.min())

    def recall(self, x, max_iter=20, x_known=None):
        """Bidirectional iteration x -> y -> x' -> y' ... until stable (resonance).
        x may contain 0 for unknown pixels; x_known (bool mask) clamps known pixels."""
        x = x.copy()
        y = self.forward(x)
        for it in range(1, max_iter + 1):
            x_new = self.backward(y, prev=np.where(x == 0, 1, x))
            if x_known is not None:
                x_new = np.where(x_known, x, x_new)
            y_new = self.forward(x_new, prev=y)
            if np.array_equal(x_new, x) and np.array_equal(y_new, y):
                return x_new, y_new, it
            x, y = x_new, y_new
        return x, y, max_iter

    def classify(self, x):
        return self.decode(self.forward(x))[0]


# ----------------------------------------------------------------------------
# application
# ----------------------------------------------------------------------------
def ask_int(msg, lo, hi, default=None):
    while True:
        s = input(f"{msg} [{lo}-{hi}]" + (f" (default {default})" if default is not None else "") + ": ").strip()
        if s == "" and default is not None: return default
        try:
            v = int(s)
            if lo <= v <= hi: return v
        except ValueError:
            pass
        print("  invalid input.")


class App:
    def __init__(self, X, y, source):
        self.source = source
        self.rng = np.random.default_rng(SEED)
        self.Xb_all, self.y_all = to_bipolar(X), y
        self.codes = make_codes()
        self.bam = BAM(self.codes)
        self.stored = False
        self.per_class = 0
        self.tr_idx = self.te_idx = None
        self.split(per_class=10)

    # choose stored (train) / unseen (test) samples
    def split(self, per_class):
        tr = []
        for d in range(10):
            idx = np.where(self.y_all == d)[0]
            tr += list(self.rng.permutation(idx)[:per_class])
        self.tr_idx = np.array(sorted(tr))
        self.te_idx = np.setdiff1d(np.arange(len(self.y_all)), self.tr_idx)
        self.per_class = per_class

    def need_store(self):
        if not self.stored:
            print("Nothing stored yet - storing with default settings (option 1) first.")
            self.op1(default=True)

    def acc(self, idx, noise=None, mask_fn=None):
        ok = 0
        for i in idx:
            x = self.Xb_all[i].copy()
            if noise: x = self.add_noise(x, noise)
            ok += self.bam.classify(x) == self.y_all[i]
        return ok / len(idx)

    def add_noise(self, x, frac):
        x = x.copy()
        k = int(round(frac * 64))
        pos = self.rng.choice(64, k, replace=False)
        x[pos] *= -1
        return x

    def pick_sample(self, pool="test"):
        d = ask_int("Digit class", 0, 9)
        idx = self.te_idx if pool == "test" else self.tr_idx
        cand = idx[self.y_all[idx] == d]
        s = input(f"  Sample row number (blank = random {pool} sample of digit {d}): ").strip()
        if s.isdigit() and int(s) < len(self.y_all): return int(s)
        return int(self.rng.choice(cand))

    # -- 1
    def op1(self, default=False):
        k = 10 if default else ask_int("Training pairs to store per digit", 1, 100, 10)
        self.split(k)
        self.bam.store(self.Xb_all[self.tr_idx], self.y_all[self.tr_idx])
        self.stored = True
        print(f"\nOutput codes ({CODE_LEN}-bit bipolar, mutually orthogonal):")
        for d in range(10):
            print(f"  digit {d}: {code_str(self.codes[d])}")
        print(f"\nStored {self.bam.n_pairs} pattern pairs ({k} per digit) with W = sum (x-mu)^T y")
        print(f"Weight matrix shape: {self.bam.W.shape}   (|w| max = {np.abs(self.bam.W).max():.1f})")
        print(f"Recall accuracy on STORED pairs : {self.acc(self.tr_idx)*100:.1f}%")
        print(f"Recall accuracy on UNSEEN digits: {self.acc(self.te_idx)*100:.1f}%  ({len(self.te_idx)} samples)")

    # -- 2
    def op2(self):
        self.need_store()
        i = self.pick_sample("test")
        x = self.Xb_all[i]
        y = self.bam.forward(x)
        d, dist = self.bam.decode(y)
        show(x, f"\nInput pattern (row {i}, true digit {self.y_all[i]}):")
        print(f"\nRecalled code : {code_str(y)}")
        print(f"Nearest code  : digit {d}  (Hamming distance {dist})")
        print(f"Result        : {'CORRECT' if d == self.y_all[i] else 'WRONG'}")
        net = self.bam._center(x) @ self.bam.W
        print("Net input per code bit: " + " ".join(f"{v:+.0f}" for v in net))

    # -- 3
    def op3(self):
        self.need_store()
        d = ask_int("Digit code to present", 0, 9)
        y = self.codes[d]
        x = self.bam.backward(y)
        proto = np.where(self.Xb_all[self.y_all == d].mean(0) > 0, 1, -1)
        print(f"\nCode for digit {d}: {code_str(y)}")
        side_by_side(x, proto, "Reconstructed", "Class mean image")
        match = (x == proto).mean() * 100
        print(f"\nPixel agreement with class mean image: {match:.1f}%")
        print(f"Forward check of the reconstruction decodes to digit {self.bam.classify(x)}")

    # -- 4
    def op4(self):
        self.need_store()
        print(f"{'Noise':>7}{'Stored acc':>12}{'Unseen acc':>12}")
        for f in (0, 0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50):
            print(f"{f*100:>6.0f}%{self.acc(self.tr_idx, f)*100:>11.1f}%{self.acc(self.te_idx, f)*100:>11.1f}%")
        f = float(input("\nNoise fraction for a visual demo (0-1, blank=0.2): ") or 0.2)
        i = self.pick_sample("train")
        clean = self.Xb_all[i]
        noisy = self.add_noise(clean, f)
        x, y, it = self.bam.recall(noisy)
        d, dist = self.bam.decode(y)
        side_by_side(noisy, x, "Noisy input", "BAM output")
        print(f"\nTrue digit {self.y_all[i]}, recalled digit {d} (distance {dist}), iterations {it}")
        print(f"Pixels flipped by noise: {(noisy != clean).sum()}, still wrong after recall: {(x != clean).sum()}")

    # -- 5
    def op5(self):
        self.need_store()
        print("Hide part of the image:\n 1) top half  2) bottom half  3) left half  4) right half  5) random pixels")
        m = ask_int("Mode", 1, 5, 2)
        keep = np.ones((8, 8), bool)
        if m == 1: keep[:4, :] = False
        elif m == 2: keep[4:, :] = False
        elif m == 3: keep[:, :4] = False
        elif m == 4: keep[:, 4:] = False
        else:
            keep = self.rng.random((8, 8)) > 0.5
        keep = keep.ravel()

        def partial(x): return np.where(keep, x, 0)

        i = self.pick_sample("test")
        x_full = self.Xb_all[i]
        xp = partial(x_full)
        x_out, y, it = self.bam.recall(xp, x_known=keep)
        d, dist = self.bam.decode(y)
        show(xp, f"\nPartial input (row {i}, true digit {self.y_all[i]}); blank = unknown:")
        side_by_side(x_out, x_full, "Completed", "Original")
        hidden = ~keep
        print(f"\nRecalled digit {d} (distance {dist}), iterations {it}")
        print(f"Hidden pixels restored correctly: {(x_out[hidden] == x_full[hidden]).mean()*100:.1f}%")
        ok = np.mean([self.bam.decode(self.bam.recall(partial(self.Xb_all[j]), x_known=keep)[1])[0]
                      == self.y_all[j] for j in self.te_idx[:300]])
        print(f"Digit-identification accuracy with this mask over 300 unseen samples: {ok*100:.1f}%")

    # -- 6
    def op6(self):
        self.need_store()
        for name, idx in (("STORED", self.tr_idx), ("UNSEEN", self.te_idx[:500])):
            stable = same = 0
            its = []
            for i in idx:
                x0 = self.Xb_all[i]
                y0 = self.bam.forward(x0)
                x1 = self.bam.backward(y0)
                y1 = self.bam.forward(x1)
                stable += np.array_equal(y0, y1)               # code survives a round trip
                same += self.bam.decode(y0)[0] == self.bam.decode(y1)[0]
                its.append(self.bam.recall(x0)[2])
            n = len(idx)
            print(f"[{name}] round-trip code identical: {stable/n*100:.1f}%  "
                  f"same decoded digit: {same/n*100:.1f}%  mean iterations to resonance: {np.mean(its):.2f}")
        print("\nCode -> pattern -> code consistency for each digit:")
        for d in range(10):
            x = self.bam.backward(self.codes[d]); y = self.bam.forward(x)
            print(f"  digit {d}: {'consistent' if np.array_equal(y, self.codes[d]) else 'INCONSISTENT'}"
                  f"  (hamming drift {(y != self.codes[d]).sum()})")

    # -- 7
    def op7(self):
        print("Capacity test: store k pairs per digit, test on the same unseen set\n")
        old = (self.per_class, self.stored)
        base_te = None
        print(f"{'k/digit':>8}{'pairs':>7}{'stored acc':>12}{'unseen acc':>12}{'noisy20% acc':>14}")
        for k in (1, 2, 3, 5, 10, 20, 40, 80):
            self.split(k)
            self.bam.store(self.Xb_all[self.tr_idx], self.y_all[self.tr_idx])
            print(f"{k:>8}{self.bam.n_pairs:>7}{self.acc(self.tr_idx)*100:>11.1f}%"
                  f"{self.acc(self.te_idx[:400])*100:>11.1f}%{self.acc(self.te_idx[:400], 0.2)*100:>13.1f}%")
        print("\nRandom-code capacity probe (distinct patterns stored with random bipolar codes):")
        for p in (4, 8, 16, 32, 64):
            P = self.rng.choice([-1, 1], (p, 64)); Y = self.rng.choice([-1, 1], (p, CODE_LEN))
            W = sum(np.outer(a, b) for a, b in zip(P, Y))
            rec = np.mean([np.array_equal(sgn(a @ W), b) for a, b in zip(P, Y)])
            print(f"  {p:>3} random pairs -> exact recall {rec*100:5.1f}%")
        self.split(old[0])
        self.bam.store(self.Xb_all[self.tr_idx], self.y_all[self.tr_idx]); self.stored = True
        print(f"\n(Network restored to {old[0]} pairs per digit.)")

    # -- 8
    def op8(self):
        self.need_store()
        C = np.zeros((10, 10), int)
        for i in self.te_idx:
            C[self.y_all[i], self.bam.classify(self.Xb_all[i])] += 1
        print("Confusion matrix (rows = true digit, columns = recalled digit)")
        print("      " + "".join(f"{d:>5}" for d in range(10)))
        for d in range(10):
            print(f"  {d:>3} " + "".join(f"{C[d, e]:>5}" for e in range(10)))
        errs = [(C[a, b], a, b) for a in range(10) for b in range(10) if a != b and C[a, b]]
        errs.sort(reverse=True)
        print("\nTop cross-association errors (true -> recalled):")
        for n, a, b in errs[:6]:
            print(f"  {a} -> {b}: {n} cases   (pattern-prototype overlap "
                  f"{self.proto_overlap(a, b):+.2f}, code Hamming distance {(self.codes[a] != self.codes[b]).sum()})")
        if not errs: print("  none")
        # codes separation
        H = [(self.codes[a] != self.codes[b]).sum() for a in range(10) for b in range(a + 1, 10)]
        print(f"\nCode separation: min Hamming distance {min(H)}, max {max(H)} -> codes are well separated.")
        print("Most error-prone digits are those whose input patterns overlap heavily (e.g. 3/5/8, 4/9, 1/7).")

    def proto_overlap(self, a, b):
        pa = self.Xb_all[self.y_all == a].mean(0); pb = self.Xb_all[self.y_all == b].mean(0)
        return float(np.corrcoef(pa, pb)[0, 1])

    # -- 9
    def op9(self):
        self.need_store()
        print("=" * 62); print("BAM ASSOCIATIVE MEMORY REPORT"); print("=" * 62)
        print(f"Dataset        : {self.source}  ({len(self.y_all)} samples)")
        print(f"Input          : 64 bipolar pixels (>= {THRESHOLD} -> +1, else -1)")
        print(f"Output code    : {CODE_LEN}-bit orthogonal bipolar, 10 classes")
        print(f"Stored pairs   : {self.bam.n_pairs} ({self.per_class} per digit)   W shape {self.bam.W.shape}")
        print(f"Stored recall  : {self.acc(self.tr_idx)*100:.1f}%")
        print(f"Unseen recall  : {self.acc(self.te_idx)*100:.1f}% on {len(self.te_idx)} samples")
        for f in (0.1, 0.2, 0.3):
            print(f"  with {int(f*100)}% pixel noise: {self.acc(self.te_idx[:500], f)*100:.1f}%")
        per = [np.mean([self.bam.classify(self.Xb_all[i]) == d for i in self.te_idx[self.y_all[self.te_idx] == d]])
               for d in range(10)]
        print("Per-digit unseen accuracy: " + "  ".join(f"{d}:{p*100:.0f}%" for d, p in enumerate(per)))
        ok = sum(np.array_equal(self.bam.forward(self.bam.backward(self.codes[d])), self.codes[d]) for d in range(10))
        print(f"Code->pattern->code consistent for {ok}/10 digits")


def main():
    X, y, source = load_data(sys.argv[1] if len(sys.argv) > 1 else None)
    print(f"[data] {len(y)} samples loaded.\n")
    app = App(X, y, source)
    app.op1(default=True)
    ops = {"1": app.op1, "2": app.op2, "3": app.op3, "4": app.op4, "5": app.op5,
           "6": app.op6, "7": app.op7, "8": app.op8, "9": app.op9}
    while True:
        print("\n===== BAM DIGIT-CODE ASSOCIATION SYSTEM =====")
        print("1. Store Digit-Code Pattern Associations")
        print("2. Forward Digit Recall Challenge")
        print("3. Reverse Digit Reconstruction Challenge")
        print("4. Noisy Pattern Association Test")
        print("5. Partial Digit Completion Challenge")
        print("6. Bidirectional Recall Consistency Analyzer")
        print("7. Association Capacity Challenge")
        print("8. Cross-Association Error Detector")
        print("9. BAM Associative Memory Report")
        print("10. Exit")
        try:
            c = input("Choice: ").strip()
        except EOFError:
            break
        if c == "10":
            print("Goodbye."); break
        if c in ops:
            print(); ops[c]()
        else:
            print("Invalid choice.")


if __name__ == "__main__":
    main()

[data] Using scikit-learn bundled UCI optdigits (1797 samples).
[data] 1797 samples loaded.


Output codes (16-bit bipolar, mutually orthogonal):
  digit 0: +-+-+-+-+-+-+-+-
  digit 1: ++--++--++--++--
  digit 2: +--++--++--++--+
  digit 3: ++++----++++----
  digit 4: +-+--+-++-+--+-+
  digit 5: ++----++++----++
  digit 6: +--+-++-+--+-++-
  digit 7: ++++++++--------
  digit 8: +-+-+-+--+-+-+-+
  digit 9: ++--++----++--++

Stored 100 pattern pairs (10 per digit) with W = sum (x-mu)^T y
Weight matrix shape: (64, 16)   (|w| max = 64.0)
Recall accuracy on STORED pairs : 83.0%
Recall accuracy on UNSEEN digits: 70.2%  (1697 samples)

===== BAM DIGIT-CODE ASSOCIATION SYSTEM =====
1. Store Digit-Code Pattern Associations
2. Forward Digit Recall Challenge
3. Reverse Digit Reconstruction Challenge
4. Noisy Pattern Association Test
5. Partial Digit Completion Challenge
6. Bidirectional Recall Consistency Analyzer
7. Association Capacity Challenge
8. Cross-Association Error Detector
9. BAM Associ